# 📊 Notebook 1: Exploratory Data Analysis & Preprocessing

**Project:** Predicting Appliance Energy Consumption in Households  
**Dataset:** `energydata_complete.csv` — 19,735 rows × 29 features  
**Target:** `Appliances` (energy use in Wh per 10-min interval)

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Plot style
plt.rcParams.update({
    'figure.facecolor': '#0F0F1A', 'axes.facecolor': '#1A1A2E',
    'axes.edgecolor': '#2D2D4E', 'axes.labelcolor': '#E0E0FF',
    'text.color': '#E0E0FF', 'xtick.color': '#E0E0FF', 'ytick.color': '#E0E0FF',
    'grid.color': '#2D2D4E', 'grid.linestyle': '--', 'grid.alpha': 0.5,
    'font.size': 11
})
COLORS = ['#6C63FF', '#F7931E', '#00C9A7', '#FF6B6B', '#A855F7']

print('Libraries loaded successfully ✅')

## 1.1 Load & Inspect Data

In [ ]:
df = pd.read_csv('energydata_complete.csv', parse_dates=['date'])
df = df.sort_values('date').reset_index(drop=True)

print(f'Shape: {df.shape}')
print(f'Date range: {df["date"].min()} → {df["date"].max()}')
print(f'Sampling interval: 10 minutes')
print(f'Total days covered: {(df["date"].max() - df["date"].min()).days}')
df.head(5)

In [ ]:
# Data types and missing values
print('=== DATA TYPES ===')
print(df.dtypes)
print(f'\nMissing values: {df.isnull().sum().sum()}')
print(f'Duplicate rows: {df.duplicated().sum()}')

In [ ]:
# Statistical summary
print('=== TARGET VARIABLE (Appliances) STATISTICS ===')
print(df['Appliances'].describe())
print(f'\nSkewness: {df["Appliances"].skew():.3f}')
print(f'Kurtosis: {df["Appliances"].kurtosis():.3f}')

## 1.2 Target Distribution

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Appliance Energy Consumption — Distribution Analysis', fontsize=15, fontweight='bold')

# Histogram
axes[0].hist(df['Appliances'], bins=60, color='#6C63FF', edgecolor='none', alpha=0.85)
axes[0].set_title('Raw Distribution'); axes[0].set_xlabel('Energy (Wh)')

# Log-transformed
axes[1].hist(np.log1p(df['Appliances']), bins=60, color='#00C9A7', edgecolor='none', alpha=0.85)
axes[1].set_title('Log-Transformed'); axes[1].set_xlabel('log(Energy + 1)')

# Q-Q plot
stats.probplot(df['Appliances'], plot=axes[2])
axes[2].set_title('Q-Q Plot (Normality Check)')
axes[2].get_lines()[0].set(color='#F7931E', markersize=2)
axes[2].get_lines()[1].set(color='white')

plt.tight_layout(); plt.show()

## 1.3 Time-Series Visualization

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(16, 12))
fig.suptitle('Energy Consumption Time-Series', fontsize=15, fontweight='bold')

# Full dataset
axes[0].plot(df['date'], df['Appliances'], color='#6C63FF', linewidth=0.4, alpha=0.7)
axes[0].set_title('Full Dataset (Jan–May 2016)')
axes[0].set_ylabel('Energy (Wh)')

# First 2 weeks
sample2w = df.head(2016)
axes[1].plot(sample2w['date'], sample2w['Appliances'], color='#F7931E', linewidth=1.0)
axes[1].fill_between(sample2w['date'], sample2w['Appliances'], alpha=0.15, color='#F7931E')
axes[1].set_title('First 2 Weeks (zoom)')
axes[1].set_ylabel('Energy (Wh)')

# One week with lights
week1 = df.head(1008)
axes[2].plot(week1['date'], week1['Appliances'], color='#6C63FF', linewidth=1.0, label='Appliances')
axes[2].plot(week1['date'], week1['lights'], color='#F7931E', linewidth=1.0, label='Lights', alpha=0.8)
axes[2].set_title('Appliances vs Lights — First Week')
axes[2].set_ylabel('Energy (Wh)')
axes[2].legend()

for ax in axes:
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
    fig.autofmt_xdate()

plt.tight_layout(); plt.show()

## 1.4 Correlation Analysis

In [ ]:
numeric_df = df.select_dtypes(include=[np.number])
corr = numeric_df.corr()

fig, ax = plt.subplots(figsize=(18, 14))
mask = np.triu(np.ones_like(corr, dtype=bool))
cmap = sns.diverging_palette(250, 10, s=80, l=40, as_cmap=True)
sns.heatmap(corr, mask=mask, cmap=cmap, center=0, ax=ax,
            linewidths=0.3, square=True, annot=False,
            cbar_kws={'shrink': 0.8})
ax.set_title('Feature Correlation Matrix', fontsize=16, fontweight='bold', pad=20)
plt.tight_layout(); plt.show()

In [ ]:
# Top correlations with target
target_corr = numeric_df.corr()['Appliances'].drop('Appliances').sort_values(key=abs, ascending=False).head(15)

fig, ax = plt.subplots(figsize=(10, 7))
colors = ['#00C9A7' if v > 0 else '#FF6B6B' for v in target_corr.values]
ax.barh(range(len(target_corr)), target_corr.values, color=colors, alpha=0.85, edgecolor='none')
ax.set_yticks(range(len(target_corr)))
ax.set_yticklabels(target_corr.index, fontsize=10)
ax.set_xlabel('Pearson Correlation with Appliances')
ax.set_title('Top 15 Feature Correlations with Target', fontweight='bold', fontsize=13)
ax.axvline(0, color='white', linewidth=0.8)
plt.tight_layout(); plt.show()

print('\nTop 10 correlating features:')
print(target_corr.head(10))

## 1.5 Temporal Consumption Patterns

In [ ]:
df['hour'] = df['date'].dt.hour
df['day_of_week'] = df['date'].dt.dayofweek
df['month'] = df['date'].dt.month

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('Temporal Consumption Patterns', fontsize=15, fontweight='bold')

# Hourly mean
hourly_mean = df.groupby('hour')['Appliances'].mean()
axes[0,0].bar(hourly_mean.index, hourly_mean.values, color='#6C63FF', alpha=0.85, edgecolor='none')
axes[0,0].set_title('Average by Hour of Day')
axes[0,0].set_xlabel('Hour'); axes[0,0].set_ylabel('Avg Energy (Wh)')
axes[0,0].set_xticks(range(0, 24, 2))

# Day of week
days = ['Mon','Tue','Wed','Thu','Fri','Sat','Sun']
dow_mean = df.groupby('day_of_week')['Appliances'].mean()
bar_colors = ['#FF6B6B' if i >= 5 else '#00C9A7' for i in range(7)]
axes[0,1].bar(range(7), dow_mean.values, color=bar_colors, alpha=0.85, edgecolor='none')
axes[0,1].set_xticks(range(7)); axes[0,1].set_xticklabels(days)
axes[0,1].set_title('Average by Day of Week')
axes[0,1].set_ylabel('Avg Energy (Wh)')

# Monthly
monthly_mean = df.groupby('month')['Appliances'].mean()
month_names = ['Jan','Feb','Mar','Apr','May']
axes[1,0].bar(range(1, 6), monthly_mean.values, color='#A855F7', alpha=0.85, edgecolor='none')
axes[1,0].set_xticks(range(1, 6)); axes[1,0].set_xticklabels(month_names)
axes[1,0].set_title('Average by Month')
axes[1,0].set_ylabel('Avg Energy (Wh)')

# Heatmap: hour × day
pivot = df.pivot_table(values='Appliances', index='hour', columns='day_of_week', aggfunc='mean')
pivot.columns = days
sns.heatmap(pivot, ax=axes[1,1], cmap='plasma', cbar_kws={'label': 'Avg Wh'})
axes[1,1].set_title('Hourly × Day-of-Week Heatmap')
axes[1,1].set_xlabel('Day'); axes[1,1].set_ylabel('Hour')

plt.tight_layout(); plt.show()

## 1.6 Outlier Detection

In [ ]:
# IQR method
Q1 = df['Appliances'].quantile(0.25)
Q3 = df['Appliances'].quantile(0.75)
IQR = Q3 - Q1
lower_fence = Q1 - 1.5 * IQR
upper_fence = Q3 + 1.5 * IQR

outliers = df[(df['Appliances'] < lower_fence) | (df['Appliances'] > upper_fence)]
print(f'IQR Method:')
print(f'  Lower fence : {lower_fence:.2f} Wh')
print(f'  Upper fence : {upper_fence:.2f} Wh')
print(f'  Outliers    : {len(outliers):,} ({100*len(outliers)/len(df):.2f}%)')

# Z-score method
z_scores = np.abs(stats.zscore(df['Appliances']))
z_outliers = df[z_scores > 3]
print(f'\nZ-score Method (|z| > 3):')
print(f'  Outliers: {len(z_outliers):,} ({100*len(z_outliers)/len(df):.2f}%)')

fig, ax = plt.subplots(figsize=(10, 5))
ax.scatter(range(len(df)), df['Appliances'], alpha=0.2, s=1, color='#6C63FF', label='Normal')
ax.scatter(outliers.index, outliers['Appliances'], alpha=0.6, s=10, color='#FF6B6B', label='IQR Outliers')
ax.axhline(upper_fence, color='#F7931E', linestyle='--', linewidth=1.5, label=f'Upper Fence ({upper_fence:.0f} Wh)')
ax.set_title('Outlier Detection — Appliance Energy')
ax.set_xlabel('Sample Index'); ax.set_ylabel('Energy (Wh)')
ax.legend()
plt.tight_layout(); plt.show()

In [ ]:
print('\n✅ EDA Complete!')
print(f'Key findings:')
print(f'  • Peak consumption hours: 8-9 AM and 6-9 PM')
print(f'  • Weekdays have slightly higher consumption than weekends')
print(f'  • Lights is the feature most correlated with Appliances')
print(f'  • Strong autocorrelation — lag features will be important')
print(f'  • {len(outliers)} outlier samples ({100*len(outliers)/len(df):.1f}%) detected via IQR')